# How to use string templates

Python's `string.Template` class provides a simple and safe way to perform string substitution. Unlike f-strings or `str.format()`, templates are designed for situations where the format string comes from an untrusted source, such as user input or a configuration file.

This guide covers the basics of template substitution, explains why templates are safer than other formatting methods for user-provided strings, and helps you decide which approach to use in different situations.

## Basic template substitution

The `string.Template` class uses `$`-based placeholders for variable substitution. You create a template with placeholder names, then call a method to fill in the values.

In [ ]:
from string import Template

# Create a template with $variable placeholders
greeting = Template("Hello, $name! Welcome to $city.")

# Substitute values using keyword arguments
result = greeting.substitute(name="Alice", city="London")
print(result)

# You can also pass a dictionary
data = {"name": "Bob", "city": "Manchester"}
result = greeting.substitute(data)
print(result)

## `substitute()` vs `safe_substitute()`

The `Template` class provides two substitution methods:

- **`substitute()`** raises a `KeyError` if any placeholder is missing from the provided values
- **`safe_substitute()`** leaves missing placeholders unchanged instead of raising an error

Use `substitute()` when all values must be provided. Use `safe_substitute()` when partial substitution is acceptable.

In [ ]:
from string import Template

template = Template("Dear $title $surname, your order #$order_id is ready.")

# substitute() with all values provided works fine
print(template.substitute(title="Dr", surname="Smith", order_id="12345"))

In [ ]:
from string import Template

template = Template("Dear $title $surname, your order #$order_id is ready.")

# substitute() raises KeyError when a value is missing
try:
    result = template.substitute(title="Dr", surname="Smith")
except KeyError as error:
    print(f"KeyError: {error}")

In [ ]:
from string import Template

template = Template("Dear $title $surname, your order #$order_id is ready.")

# safe_substitute() leaves the missing placeholder in place
result = template.safe_substitute(title="Dr", surname="Smith")
print(result)

## Why templates are safer than f-strings and `format()` for user input

When a format string comes from user input, `str.format()` can be exploited to access object attributes and potentially leak sensitive data. Templates do not have this vulnerability because they only support simple name-based substitution.

The following example demonstrates the difference.

In [ ]:
from string import Template

# Imagine a user provides a format string for a message
# A malicious user might try to access object attributes

secret_data = {"api_key": "sk-secret-12345", "name": "Alice"}

# With str.format(), a user-provided string can access attributes
malicious_format = "{name} {api_key}"
print("str.format() result:", malicious_format.format(**secret_data))

# With str.format_map(), it is even possible to access object internals
# For example: "{name.__class__}" could reveal implementation details

# With Template, only simple $name substitution is supported
# There is no way to access attributes, call methods, or evaluate expressions
safe_template = Template("$name")
print("Template result:", safe_template.substitute(name="Alice"))

## Template syntax details

The `Template` class supports several placeholder formats:

| Syntax | Meaning |
|---|---|
| `$identifier` | Simple placeholder, substituted with the named value |
| `${identifier}` | Braced placeholder, useful when the placeholder is adjacent to other text |
| `$$` | An escaped dollar sign, produces a literal `$` in the output |

In [ ]:
from string import Template

# Simple $identifier
t1 = Template("Hello, $name!")
print(t1.substitute(name="Alice"))

# Braced ${identifier} to avoid ambiguity
t2 = Template("The ${item}s are on sale.")  # Without braces, it would look for $items
print(t2.substitute(item="book"))

# Escaped $$ for a literal dollar sign
t3 = Template("The price is $$${price} (approximately £$pounds).")
print(t3.substitute(price="49.99", pounds="39.50"))

## Practical example: personalised messages

A common use case for templates is generating personalised messages from a user-defined template string. This allows non-technical users to define message formats without writing Python code.

In [ ]:
from string import Template


def send_personalised_messages(
    template_text: str,
    recipients: list[dict],
) -> list[str]:
    """Generate personalised messages from a template and a list of recipients.

    Uses safe_substitute so that missing fields do not cause errors.

    Args:
        template_text: A template string with $variable placeholders.
        recipients: A list of dictionaries, each containing values for the placeholders.

    Returns:
        A list of personalised message strings.
    """
    template = Template(template_text)
    messages = []
    for recipient in recipients:
        message = template.safe_substitute(recipient)
        messages.append(message)
    return messages


# A user-defined template (could come from a configuration file or form input)
user_template = "Dear $name, your appointment is on $date at $time. Reference: $ref"

recipients = [
    {"name": "Alice", "date": "15/03/2025", "time": "09:30", "ref": "APT-001"},
    {"name": "Bob", "date": "16/03/2025", "time": "14:00", "ref": "APT-002"},
    {"name": "Charlie", "date": "17/03/2025", "time": "11:15"},  # Missing 'ref'
]

messages = send_personalised_messages(user_template, recipients)
for message in messages:
    print(message)
    print()

## When to use templates vs f-strings vs `format()`

Each string formatting method has its strengths. Choose the right tool based on who controls the format string and what features you need.

| Method | Best for | Format string source | Expression support |
|---|---|---|---|
| **f-strings** | Most everyday formatting | Developer-written code | Full Python expressions |
| **`str.format()`** | Dynamic formatting with known, trusted templates | Trusted code or data | Attribute access, indexing, format specs |
| **`string.Template`** | User-provided format strings | Untrusted or external input | Simple name substitution only |

**General guidance:**

- Use **f-strings** when you are writing the format string directly in your code. They are the most readable and the most powerful.
- Use **`str.format()`** when you need to store a format string as a variable or in a configuration file, and you trust the source.
- Use **`string.Template`** when the format string comes from user input, an external file, or any untrusted source. The limited substitution syntax prevents accidental or malicious access to Python internals.

## Summary

In this guide, you learned how to:

- Create templates using `string.Template` with `$variable` placeholders
- Use `substitute()` for strict substitution and `safe_substitute()` for lenient substitution
- Understand why templates are safer than `str.format()` for user-provided format strings
- Use braced placeholders (`${identifier}`) and escaped dollar signs (`$$`)
- Build a practical personalised messaging system using templates
- Choose between f-strings, `str.format()`, and `string.Template` for different use cases